# 07 - ChromaDB Indexing

## Objective

Create a ChromaDB vector store for Pipeline 1.

Pipeline:

Documents
→ Recursive Chunking
→ MiniLM Embeddings
→ ChromaDB
→ Qwen3:14B

## Outcome

- Build Chroma vector index
- Store chunk embeddings
- Validate vector count
- Test similarity search

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Load Source Documents

In [2]:
from src.config import DOCS_PATH
from src.loaders.document_loader import DocumentLoader

loader = DocumentLoader(DOCS_PATH)

documents = loader.load()

print(
    f"Source Pages: {len(documents):,}"
)

Starting Document Loading...

Loading PDF : iSupplier_portal.pdf

Loading PDF : Purchasing_manual.pdf


DOCUMENT LOADING SUMMARY
PDF Files Loaded   : 2
DOCX Files Loaded  : 0
XLSX Files Loaded  : 0
Total Documents    : 1502
Source Pages: 1,502


## Create Recursive Chunks

Pipeline 1 uses Recursive Character Chunking.

In [3]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

recursive_chunks = (
    recursive_splitter.split_documents(
        documents
    )
)

print(
    f"Chunks Created: {len(recursive_chunks):,}"
)

Chunks Created: 3,519


## Detect Available Hardware

In [4]:
import torch

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    f"Embedding Device: {device}"
)

if torch.cuda.is_available():

    print(
        f"GPU : {torch.cuda.get_device_name(0)}"
    )

Embedding Device: cuda
GPU : NVIDIA RTX A5000 Laptop GPU


## Load MiniLM Embedding Model

In [5]:
from langchain_huggingface import (
    HuggingFaceEmbeddings
)

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={
        "device": device
    }
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Create ChromaDB Vector Store

In [6]:
from pathlib import Path

chroma_dir = Path(
    "../vectorstores/chroma_minilm"
)

chroma_dir.mkdir(
    parents=True,
    exist_ok=True
)

print(
    f"Chroma Directory: {chroma_dir.resolve()}"
)

Chroma Directory: D:\AI\Projects\oracle-rag-comparison-lab\vectorstores\chroma_minilm


In [7]:
from langchain_chroma import Chroma

In [8]:
chroma_db = Chroma.from_documents(
    documents=recursive_chunks,
    embedding=embedding_model,
    persist_directory=str(chroma_dir)
)

## Validate Vector Store Creation

In [9]:
collection = chroma_db._collection

print("=" * 80)

print("CHROMADB VALIDATION")

print("=" * 80)

print(
    f"Vector Count: {collection.count():,}"
)

CHROMADB VALIDATION
Vector Count: 3,519


## Test Similarity Search

In [10]:
query = (
    "How do I create a purchase requisition?"
)

results = chroma_db.similarity_search(
    query,
    k=3
)

In [11]:
print("=" * 80)

print("SIMILARITY SEARCH RESULTS")

print("=" * 80)

print(
    f"Results Returned: {len(results)}"
)

SIMILARITY SEARCH RESULTS
Results Returned: 3


In [12]:
for idx, doc in enumerate(results):

    print("\n" + "=" * 80)

    print(
        f"RESULT #{idx+1}"
    )

    print("=" * 80)

    print(
        doc.metadata
    )

    print("\n")

    print(
        doc.page_content[:1000]
    )


RESULT #1
{'file_type': 'pdf', 'source': 'Purchasing_manual.pdf', 'page': 231}


system. These people can create requisitions for anyone in the company by identifying 
the appropriate requestor directly on the requisition line. You might also want to limit 
some preparers to internal requisitions and others to purchase requisitions. You can 
also print requisitions to obtain signature approvals if some of the approvers do not 
have access to the system. You can then assign an individual to update the 
authorization status for these requisitions.
Purchase Requisitions
Use the Requisitions window to create requisitions. You must choose the requisition 
type (internal or purchase). You can also provide a description, unlimited notes, and 
defaults for requisition lines. For each requisition line, you choose the item you want to 
order along with the quantity and delivery location. You can get sourced pricing from 
catalog quotations or open blanket purchase agreements. You can also choos

In [14]:
for idx, doc in enumerate(results):

    print("\n" + "=" * 80)

    print(
        f"RESULT #{idx+1}"
    )

    print("=" * 80)

    print(
        doc.metadata
    )

    print("\n")

    print(
        doc.page_content[:1000]
    )


RESULT #1
{'file_type': 'pdf', 'source': 'Purchasing_manual.pdf', 'page': 231}


system. These people can create requisitions for anyone in the company by identifying 
the appropriate requestor directly on the requisition line. You might also want to limit 
some preparers to internal requisitions and others to purchase requisitions. You can 
also print requisitions to obtain signature approvals if some of the approvers do not 
have access to the system. You can then assign an individual to update the 
authorization status for these requisitions.
Purchase Requisitions
Use the Requisitions window to create requisitions. You must choose the requisition 
type (internal or purchase). You can also provide a description, unlimited notes, and 
defaults for requisition lines. For each requisition line, you choose the item you want to 
order along with the quantity and delivery location. You can get sourced pricing from 
catalog quotations or open blanket purchase agreements. You can also choos

In [15]:
for idx, doc in enumerate(results):

    print("\n" + "=" * 80)

    print(
        f"RESULT #{idx+1}"
    )

    print("=" * 80)

    print(
        doc.metadata
    )

    print("\n")

    print(
        doc.page_content[:1000]
    )


RESULT #1
{'file_type': 'pdf', 'source': 'Purchasing_manual.pdf', 'page': 231}


system. These people can create requisitions for anyone in the company by identifying 
the appropriate requestor directly on the requisition line. You might also want to limit 
some preparers to internal requisitions and others to purchase requisitions. You can 
also print requisitions to obtain signature approvals if some of the approvers do not 
have access to the system. You can then assign an individual to update the 
authorization status for these requisitions.
Purchase Requisitions
Use the Requisitions window to create requisitions. You must choose the requisition 
type (internal or purchase). You can also provide a description, unlimited notes, and 
defaults for requisition lines. For each requisition line, you choose the item you want to 
order along with the quantity and delivery location. You can get sourced pricing from 
catalog quotations or open blanket purchase agreements. You can also choos

## Export Index Metadata

In [16]:
index_info = {
    "Vector Store":
        "ChromaDB",

    "Embedding Model":
        "MiniLM",

    "Embedding Dimension":
        384,

    "Chunk Count":
        len(recursive_chunks),

    "Device":
        device
}

In [17]:
import pandas as pd

index_df = pd.DataFrame(
    index_info.items(),
    columns=[
        "Property",
        "Value"
    ]
)

index_df

,Property,Value
0,Vector Store,ChromaDB
1,Embedding Model,MiniLM
2,Embedding Dimension,384
3,Chunk Count,3519
4,Device,cuda


In [18]:
from pathlib import Path

output_dir = Path(
    "../screenshots/vectorstores"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_file = (
    output_dir
    / "chroma_index_info.csv"
)

index_df.to_csv(
    output_file,
    index=False
)

print(
    f"Saved: {output_file.resolve()}"
)

Saved: D:\AI\Projects\oracle-rag-comparison-lab\screenshots\vectorstores\chroma_index_info.csv


# Conclusion

## ChromaDB Index

Pipeline 1 now supports:

- Recursive Character Chunking
- MiniLM Embeddings
- ChromaDB Similarity Search

## Validation

- Vector Store Created
- Embeddings Indexed
- Similarity Search Successful

## Next Step

Proceed to:

### 08_FAISS_Indexing.ipynb